In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

In [2]:
df = pd.read_csv('../data/weather_classification_data.csv')
df

,Temperature,Humidity,Wind Speed,Precipitation (%),Cloud Cover,Atmospheric Pressure,UV Index,Season,Visibility (km),Location,Weather Type
0,14.0,73,9.5,82.0,partly cloudy,1010.82,2,Winter,3.5,inland,Rainy
1,39.0,96,8.5,71.0,partly cloudy,1011.43,7,Spring,10.0,inland,Cloudy
2,30.0,64,7.0,16.0,clear,1018.72,5,Spring,5.5,mountain,Sunny
3,38.0,83,1.5,82.0,clear,1026.25,7,Spring,1.0,coastal,Sunny
4,27.0,74,17.0,66.0,overcast,990.67,1,Winter,2.5,mountain,Rainy
...,...,...,...,...,...,...,...,...,...,...,...
13195,10.0,74,14.5,71.0,overcast,1003.15,1,Summer,1.0,mountain,Rainy
13196,-1.0,76,3.5,23.0,cloudy,1067.23,1,Winter,6.0,coastal,Snowy
13197,30.0,77,5.5,28.0,overcast,1012.69,3,Autumn,9.0,coastal,Cloudy
13198,3.0,76,10.0,94.0,overcast,984.27,0,Winter,2.0,inland,Snowy


In [3]:
numerical_features = ['Temperature', 'Humidity', 'Wind Speed', 'Precipitation (%)', 'Atmospheric Pressure', 'UV Index', 'Visibility (km)']
categorical_features = ['Cloud Cover', 'Season', 'Location']
target_column = 'Weather Type'

In [4]:
X = df.drop(target_column, axis=1)
y = df[target_column]

# Encode variabel target
le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_

In [5]:
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

In [7]:
knn_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5)) # Menggunakan k=5
])

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

# Latih model
knn_model.fit(X_train, y_train)

# Buat prediksi
y_pred = knn_model.predict(X_test)

In [12]:
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=class_names, zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred)

In [13]:
print(f"Accuracy Score: {accuracy:.4f}\n")
print("Classification Report:\n", report)
print("\nConfusion Matrix:\n", conf_matrix)
print("\nWeather Type Labels (Referensi Confusion Matrix):\n", class_names)

Accuracy Score: 0.8975

Classification Report:
               precision    recall  f1-score   support

      Cloudy       0.84      0.90      0.87       990
       Rainy       0.87      0.90      0.88       990
       Snowy       0.94      0.91      0.93       990
       Sunny       0.95      0.88      0.91       990

    accuracy                           0.90      3960
   macro avg       0.90      0.90      0.90      3960
weighted avg       0.90      0.90      0.90      3960


Confusion Matrix:
 [[887  70  16  17]
 [ 55 891  29  15]
 [ 44  27 905  14]
 [ 71  36  12 871]]

Weather Type Labels (Referensi Confusion Matrix):
 ['Cloudy' 'Rainy' 'Snowy' 'Sunny']
